In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/f1_raw_2018_2021.csv')

# Create target immediately
df['Podium'] = (df['Position'] <= 3).astype(int)

# Sort — critical before any rolling calculations
df = df.sort_values(['Abbreviation', 'Year', 'Round']).reset_index(drop=True)

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

Loaded 1500 rows
Columns: ['Abbreviation', 'TeamName', 'GridPosition', 'Position', 'Points', 'Status', 'Year', 'Round', 'EventName', 'CircuitName', 'IsWet', 'SafetyCarDeployed', 'IsDNF', 'QualiGapToPole_pct', 'Podium']


In [2]:
# Average finishing position over last 3 races
df['AvgFinish_Last3'] = (
    df.groupby('Abbreviation')['Position']
      .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

# DNF rate over last 5 races  
df['DNF_Rate_Last5'] = (
    df.groupby('Abbreviation')['IsDNF']
      .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

# Podium rate over last 5 races (driver form)
df['PodiumRate_Last5'] = (
    df.groupby('Abbreviation')['Podium']
      .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

print("Driver rolling features created")
print(df[['Abbreviation', 'Year', 'Round', 
          'AvgFinish_Last3', 'DNF_Rate_Last5', 
          'PodiumRate_Last5']].head(10))

Driver rolling features created
  Abbreviation  Year  Round  AvgFinish_Last3  DNF_Rate_Last5  PodiumRate_Last5
0          ALB  2019      1              NaN             NaN               NaN
1          ALB  2019      2        14.000000             0.0               0.0
2          ALB  2019      4        11.500000             0.0               0.0
3          ALB  2019      5        11.333333             0.0               0.0
4          ALB  2019      6        10.333333             0.0               0.0
5          ALB  2019      7        10.000000             0.0               0.0
6          ALB  2019      8        12.666667             0.2               0.0
7          ALB  2019      9        14.000000             0.2               0.0
8          ALB  2019     10        16.333333             0.2               0.0
9          ALB  2019     11        14.000000             0.2               0.0
